# Experiment B: Supervised Buzzer Detection
**Goal:** Train classifier dengan manual labels, compare dengan unsupervised

**Prerequisites:** Manual labeling di `data/labels/top30_for_review.csv` & `random20_for_review.csv`

**Model:** Random Forest (baseline)

## 1. Setup

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Setup complete")

Setup complete


## 2. Load Labeled Data

In [ ]:
# Check if labels exist
import os
if not os.path.exists("../data/labels/top30_for_review.csv"):
    raise FileNotFoundError("Run notebook 04a first to generate label files")

# Load labeled data
top30 = pd.read_csv("../data/labels/top30_for_review.csv")
random20 = pd.read_csv("../data/labels/random20_for_review.csv")

# Combine
labeled = pd.concat([top30, random20], ignore_index=True)

# Check if labeled
if labeled["label"].isna().all() or (labeled["label"] == "").all():
    raise ValueError("No labels found! Please add labels (0 or 1) in 'label' column")

# Convert label to int
labeled["label"] = pd.to_numeric(labeled["label"], errors="coerce")
labeled = labeled.dropna(subset=["label"])
labeled["label"] = labeled["label"].astype(int)

print(f"Loaded {len(labeled)} labeled accounts")
print(f"  Buzzer: {(labeled['label'] == 1).sum()}")
print(f"  Organic: {(labeled['label'] == 0).sum()}")
print(f"\nClass balance: {(labeled['label'] == 1).sum() / len(labeled) * 100:.1f}% buzzer")

# if len(labeled) < 10:
#     raise ValueError(f"Need at least 10 labeled accounts, got {len(labeled)}")

labeled.head()

Loaded 0 labeled accounts
  Buzzer: 0
  Organic: 0

Class balance: nan% buzzer


ValueError: Need at least 10 labeled accounts, got 0

## 3. Load Full Features

In [ ]:
# Load account features with scores from 04a
accounts = pd.read_csv("../data/features/account_features_with_scores.csv")

# Merge labels
accounts = accounts.merge(labeled[["username", "label"]], on="username", how="left")

# Separate labeled and unlabeled
labeled_accounts = accounts[accounts["label"].notna()].copy()
unlabeled_accounts = accounts[accounts["label"].isna()].copy()

print(f"Total accounts: {len(accounts)}")
print(f"  Labeled: {len(labeled_accounts)}")
print(f"  Unlabeled: {len(unlabeled_accounts)}")

labeled_accounts.head()

## 4. Feature Selection & Preprocessing

In [ ]:
# Select features for training
feature_cols = [
    "n_posts", "post_rate", "avg_tags", "avg_body_len",
    "n_replies", "n_quotes", "n_originals",
    "in_degree", "out_degree", "pagerank",
    "follower_ratio", "active_hours"
]

# Check missing features
missing_features = [f for f in feature_cols if f not in labeled_accounts.columns]
if missing_features:
    raise ValueError(f"Missing features: {missing_features}. Run 04a first.")

# Prepare data
X = labeled_accounts[feature_cols].fillna(0)
y = labeled_accounts["label"].astype(int)

print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(X)}")
print(f"\nFeature list:")
for f in feature_cols:
    print(f"  - {f}")

# Check for infinite values
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"\nData ready for training")

## 5. Train-Test Split

In [ ]:
# Split with stratification
test_size = 0.2 if len(X) >= 20 else 0.3  # Adjust for small datasets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} samples")
print(f"  Buzzer: {(y_train == 1).sum()}")
print(f"  Organic: {(y_train == 0).sum()}")
print(f"\nTest set: {len(X_test)} samples")
print(f"  Buzzer: {(y_test == 1).sum()}")
print(f"  Organic: {(y_test == 0).sum()}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Features scaled")

## 6. Train Random Forest

In [ ]:
# Train model
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced'  # Handle imbalance
)

rf.fit(X_train_scaled, y_train)

# Training accuracy
train_score = rf.score(X_train_scaled, y_train)
test_score = rf.score(X_test_scaled, y_test)

print(f"Training accuracy: {train_score:.3f}")
print(f"Test accuracy: {test_score:.3f}")

# Cross-validation (if enough samples)
if len(X) >= 10:
    cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=min(5, len(X_train)), scoring='f1')
    print(f"\nCross-validation F1: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

## 7. Evaluation Metrics

In [ ]:
# Predictions
y_pred = rf.predict(X_test_scaled)
y_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["Organic", "Buzzer"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\n=== CONFUSION MATRIX ===")
print("                Predicted")
print("              Organic  Buzzer")
print(f"Actual Organic   {cm[0,0]:3d}     {cm[0,1]:3d}")
print(f"       Buzzer    {cm[1,0]:3d}     {cm[1,1]:3d}")

# ROC-AUC (if both classes in test)
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba)
    print(f"\nROC-AUC: {auc:.3f}")

## 8. Feature Importance

In [ ]:
# Feature importance
importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("=== FEATURE IMPORTANCE ===")
print(importances.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances["feature"], importances["importance"])
ax.set_xlabel("Importance")
ax.set_title("Feature Importance - Random Forest", fontsize=14, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../output/supervised_feature_importance.png", dpi=150, bbox_inches="tight")
print("\n✓ Saved: output/supervised_feature_importance.png")
plt.show()

## 9. Apply to All Accounts

In [ ]:
# Predict on all accounts
X_all = accounts[feature_cols].fillna(0).replace([np.inf, -np.inf], np.nan).fillna(0)
X_all_scaled = scaler.transform(X_all)

accounts["predicted_label"] = rf.predict(X_all_scaled)
accounts["predicted_proba"] = rf.predict_proba(X_all_scaled)[:, 1]

# Compare with unsupervised score
comparison = accounts[["username", "buzzer_score", "predicted_proba", "label"]].copy()
comparison = comparison.sort_values("predicted_proba", ascending=False)

print("=== TOP 10 BY SUPERVISED MODEL ===")
print(comparison.head(10).to_string(index=False))

# Correlation between unsupervised and supervised
corr = accounts[["buzzer_score", "predicted_proba"]].corr().iloc[0, 1]
print(f"\nCorrelation (unsupervised vs supervised): {corr:.3f}")

## 10. Comparison: Unsupervised vs Supervised

In [ ]:
# Scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

# Labeled accounts
labeled_mask = accounts["label"].notna()
buzzer_mask = (accounts["label"] == 1) & labeled_mask
organic_mask = (accounts["label"] == 0) & labeled_mask

ax.scatter(accounts.loc[organic_mask, "buzzer_score"], 
           accounts.loc[organic_mask, "predicted_proba"],
           c="green", label="Labeled Organic", s=100, alpha=0.7, edgecolors="black")
ax.scatter(accounts.loc[buzzer_mask, "buzzer_score"], 
           accounts.loc[buzzer_mask, "predicted_proba"],
           c="red", label="Labeled Buzzer", s=100, alpha=0.7, edgecolors="black")
ax.scatter(accounts.loc[~labeled_mask, "buzzer_score"], 
           accounts.loc[~labeled_mask, "predicted_proba"],
           c="gray", label="Unlabeled", s=30, alpha=0.3)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect agreement")
ax.set_xlabel("Unsupervised Score (Rule-based)", fontsize=12)
ax.set_ylabel("Supervised Probability (Random Forest)", fontsize=12)
ax.set_title(f"Unsupervised vs Supervised\nCorrelation: {corr:.3f}", 
             fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../output/unsupervised_vs_supervised.png", dpi=150, bbox_inches="tight")
print("✓ Saved: output/unsupervised_vs_supervised.png")
plt.show()

## 11. Export Results

In [ ]:
import os
os.makedirs("../data/predictions", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save predictions
predictions = accounts[["account_id", "username", "buzzer_score", 
                       "predicted_label", "predicted_proba", "label"]].copy()
predictions.to_csv("../data/predictions/all_accounts_predictions.csv", index=False)
print("✓ data/predictions/all_accounts_predictions.csv")

# Save model
import pickle
with open("../models/rf_buzzer_detector.pkl", "wb") as f:
    pickle.dump({"model": rf, "scaler": scaler, "features": feature_cols}, f)
print("✓ models/rf_buzzer_detector.pkl")

print("\n=== EXPERIMENT B COMPLETE ===")
print(f"Model trained on {len(X_train)} samples")
print(f"Test accuracy: {test_score:.3f}")
print(f"Applied to {len(accounts)} total accounts")
print("\nNext: Compare A vs B results, decide which approach for Issue 2-6")